# RAV-Med: Step 6a, LLM-as-judge and critic-style baselines
**Question.** Does ERS (a small similarity model plus an NLI model) beat what a reviewer will ask about first: simply asking a 7B instruction-tuned LLM whether the passage is relevant, supports or contradicts the claim?

**Baselines (all zero-shot, same claim and passage ERS sees; none uses a label).**
* `J_rel`, a relevance critic (the idea behind the retrieval evaluators of CRAG and Self-RAG, here as a prompt, **not** the published checkpoints).
* `J_trust`, a single "reliable evidence for this claim?" critic.
* `J_con`, a contradiction judge; `J_rel - J_con` and `J_sup - J_con` are the LLM analogues of ERS.
Each is the probability of "Yes" read from the next-token logits (one forward pass, no sampling).

**Pre-specified endpoints (unchanged from the paper).** PubMedQA: AUROC of CLEAN vs corrupted passages (cluster bootstrap over queries). HealthContradict: pairwise accuracy correct > incorrect on the top-1 window (cluster bootstrap over the 81 questions), reference claim and label-free claim. SciFact (Step 6d): AUROC of SUPPORT vs the other conditions.

**Honest design notes.**
* The judge LLM is Qwen2.5-7B-Instruct, the same family as the downstream generator of the paper. State this in the paper.
* On HealthContradict the judge scores the **window ERS selected** (most claim-similar 80 words). This favours the judge slightly (it inherits ERS's window choice). Set `JUDGE_ALL_WINDOWS = True` to score every window instead (several times slower).
* Prompts are fixed below before any result is seen. Do not edit them after looking at results; if you change one, log it as a deviation.

**Run order.** 1) set `DRY_RUN = True`, run all cells (about a minute, no GPU needed) to check that nothing breaks; 2) `DRY_RUN = False`, T4 GPU, run all. Results are cached on Google Drive, so after a disconnect just re-run.
**Uploads needed:** `rav_med_full_results_N1000.csv`, `hc_claims.csv` (both are in the repository's `data/` folder).

In [ ]:
!pip install -q sentence-transformers transformers accelerate bitsandbytes scikit-learn scipy datasets

In [ ]:
import os, re, json, random, hashlib, subprocess, shutil, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)

DRY_RUN = False        # True = tiny fake models, no GPU: checks that every cell runs in seconds. Set False for real results.
DEVICE = "cuda"
EMBED_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"   # as in the paper
NLI_MODEL   = "roberta-large-mnli"                                        # as in the paper
N_BOOT = 2000
OUT = "step6_outputs"; os.makedirs(OUT, exist_ok=True)

# ---- platform helpers: the same notebook runs on Google Colab and on Kaggle ----
KAGGLE_ROOT = "/kaggle"
ON_KAGGLE = os.path.isdir(KAGGLE_ROOT + "/working")
def _find_in_kaggle_input(name):
    for root, _, fs in os.walk(KAGGLE_ROOT + "/input"):
        if name in fs: return os.path.join(root, name)
    return None
def ensure_file(name):
    """Make `name` available in the working directory (Kaggle: from an attached dataset; Colab: upload dialog)."""
    if os.path.exists(name): return
    if ON_KAGGLE:
        p = _find_in_kaggle_input(name)
        if p is None: raise FileNotFoundError(f"{name} not found. In Kaggle: Add Input -> upload it as a dataset and attach it to this notebook.")
        shutil.copy(p, name); print("copied", name, "from", p); return
    try:
        from google.colab import files; print("Upload", name); files.upload()
    except ImportError:
        raise FileNotFoundError(name)
def load_cache(path):
    """Read a JSON cache; if the file is truncated (disconnect during a write), fall back to the .bak copy."""
    for p in (path, path + ".bak"):
        if os.path.exists(p):
            try: return json.load(open(p))
            except Exception as e: print("cache file", p, "unreadable:", e)
    return {}
def save_cache(cache, path):
    """Write atomically (temp file, then rename) and keep one backup, so a disconnect cannot corrupt the cache."""
    tmp = path + ".tmp"; json.dump(cache, open(tmp, "w"))
    if os.path.exists(path): shutil.copy(path, path + ".bak")
    os.replace(tmp, path)
def restore_cache(path):
    """Kaggle only: if a cache file from an earlier session is attached as input, copy it back."""
    if ON_KAGGLE and not os.path.exists(path):
        p = _find_in_kaggle_input(os.path.basename(path))
        if p: shutil.copy(p, path); print("restored cache from", p)

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"
LOAD_4BIT = True            # True on a free T4
N_QUERIES = 1000            # PubMedQA queries (x4 conditions); set 100 for a trial
MAX_WORDS = 250             # passage words shown to the judge
JBATCH = 8
RUN_PUBMED, RUN_HC, RUN_SCIFACT = True, True, True
MAX_RUN_HOURS = 10.5     # stop cleanly (progress saved) before a ~12 h session limit; lower it if your sessions are shorter
import time; START_TIME = time.time(); HALT = False
AUTO_DOWNLOAD_EVERY_MIN = 0   # Colab WITHOUT Drive: set to e.g. 45 so the progress file downloads to your computer every 45 min (allow multiple downloads in the browser)
RESUME_FROM_UPLOAD = False    # Colab WITHOUT Drive: set True to upload a judge_cache_*.json that you downloaded in an earlier session
JUDGE_ALL_WINDOWS = False

In [ ]:
if ON_KAGGLE:
    BASE = "/kaggle/working/rav_med"; os.makedirs(BASE, exist_ok=True)
else:
    try:
        from google.colab import drive
    except ImportError:
        drive = None; BASE = "."; print("Not on Colab: using the local folder")
    if drive is not None:
        try:
            drive.mount("/content/drive")
        except Exception as e:
            raise RuntimeError("Google Drive could not be mounted, so progress would be lost at every disconnect. Stopping on purpose. "
                               "Click the Drive icon in the left sidebar (Mount Drive), allow access in the pop-up (allow pop-ups for colab.research.google.com), then run this cell again. Original error: " + repr(e))
        BASE = "/content/drive/MyDrive/rav_med"; os.makedirs(BASE, exist_ok=True)
print("Saving progress to:", BASE)

In [ ]:
OUT = f"{BASE}/step6a_outputs"; os.makedirs(OUT, exist_ok=True)   # outputs live on Drive/Kaggle storage, so finished sections survive a disconnect
print("outputs folder:", OUT)

## 1. Scoring models (SBERT + NLI, as in the paper) and the judge LLM

In [ ]:
import torch
def _h(s): return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16) / 0xffffffff

if DRY_RUN:
    def embed(texts, bs=64):
        out = []
        for t in texts:
            v = np.random.default_rng(int(_h(t) * 1e9)).normal(size=16); out.append(v / np.linalg.norm(v))
        return np.array(out)
    def nli3(prem, hyp, bs=32):
        rows = []
        for p, h in zip(prem, hyp):
            r = np.random.default_rng(int(_h(p[:200] + "||" + h) * 1e9)).random(3); rows.append(r / r.sum())
        return np.array(rows)          # columns: contradiction, neutral, entailment
else:
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
    print("SBERT max_seq_length (as in all earlier runs):", embedder.max_seq_length)
    nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()
    _lab = {l.lower(): i for i, l in nli.config.id2label.items()}
    _IDX = [_lab["contradiction"], _lab["neutral"], _lab["entailment"]]
    def embed(texts, bs=64):
        return embedder.encode(list(texts), batch_size=bs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=len(texts) > 500)
    @torch.no_grad()
    def nli3(prem, hyp, bs=32):
        prem, hyp = list(prem), list(hyp)
        order = np.argsort([len(p) for p in prem]); out = np.zeros((len(prem), 3))
        for i in range(0, len(prem), bs):
            idx = order[i:i+bs]
            enc = nli_tok([prem[j] for j in idx], [hyp[j] for j in idx], truncation=True, padding=True, max_length=256, return_tensors="pt").to(DEVICE)
            out[idx] = torch.softmax(nli(**enc).logits.float(), -1)[:, _IDX].cpu().numpy()
        return out
def contra(prem, hyp, bs=32): return nli3(prem, hyp, bs)[:, 0]
def cos(a, b): return (embed(list(a)) * embed(list(b))).sum(1)

from sklearn.metrics import roc_auc_score
def auroc_ci(y, x, cluster, n_boot=N_BOOT, seed=SEED):
    """AUROC with a cluster bootstrap (resample whole clusters, e.g. queries or claims)."""
    y = np.asarray(y); x = np.asarray(x, float); cl = np.asarray(cluster)
    a = roc_auc_score(y, x); r = np.random.default_rng(seed); u = np.unique(cl); pos = {c: np.where(cl == c)[0] for c in u}; b = []
    for _ in range(n_boot):
        idx = np.concatenate([pos[c] for c in r.choice(u, len(u), replace=True)])
        if len(np.unique(y[idx])) == 2: b.append(roc_auc_score(y[idx], x[idx]))
    return a, np.percentile(b, 2.5), np.percentile(b, 97.5)

def win(a, b): return (np.asarray(a) > np.asarray(b)).astype(float) + 0.5 * (np.asarray(a) == np.asarray(b))
def pairwise_ci(w, topic, n_boot=N_BOOT, seed=SEED):
    """Pairwise accuracy (pooled over pairs) with a question-level cluster bootstrap, as in the HealthContradict step."""
    w = np.asarray(w, float); topic = np.asarray(topic); u = np.unique(topic); r = np.random.default_rng(seed)
    s = np.array([w[topic == t].sum() for t in u]); n = np.array([(topic == t).sum() for t in u]); bi = r.integers(0, len(u), size=(n_boot, len(u)))
    b = s[bi].sum(1) / n[bi].sum(1)
    return w.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

In [ ]:
CACHE_PATH = f"{BASE}/judge_cache_{LLM_NAME.split('/')[-1]}.json"
restore_cache(CACHE_PATH)
if RESUME_FROM_UPLOAD and not os.path.exists(CACHE_PATH):
    from google.colab import files; print("Upload your earlier judge_cache_*.json"); up = files.upload()
    for n_ in up: shutil.move(n_, CACHE_PATH)
cache = load_cache(CACHE_PATH)
print("cached judgements:", len(cache))
def _key(s, u): return hashlib.md5((LLM_NAME + "\x00" + s + "\x00" + u).encode()).hexdigest()
if DRY_RUN:
    def _score_chunk(chunk): return [_h(u) for s, u in chunk]
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    if LOAD_4BIT:
        llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, device_map={"": 0}, quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")).eval()
    else:
        llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE).eval()
    YES, NO = tok.encode("Yes", add_special_tokens=False)[0], tok.encode("No", add_special_tokens=False)[0]
    def _chat(s, u):
        try: return tok.apply_chat_template([{"role": "system", "content": s}, {"role": "user", "content": u}], tokenize=False, add_generation_prompt=True)
        except Exception: return tok.apply_chat_template([{"role": "user", "content": s + "\n\n" + u}], tokenize=False, add_generation_prompt=True)
    @torch.no_grad()
    def _score_chunk(chunk):
        try:
            enc = tok([_chat(s, u) for s, u in chunk], return_tensors="pt", padding=True).to(DEVICE)
            lg = llm(**enc).logits[:, -1, :].float()[:, [YES, NO]]
            return torch.softmax(lg, -1)[:, 0].cpu().tolist()
        except torch.cuda.OutOfMemoryError:
            import gc; gc.collect(); torch.cuda.empty_cache()
            if len(chunk) == 1: raise
            h = len(chunk) // 2; return _score_chunk(chunk[:h]) + _score_chunk(chunk[h:])

class TimeBudget(Exception): pass
def run_judge(prompts):
    global HALT
    todo = sorted({p for p in prompts if _key(*p) not in cache}, key=lambda p: len(p[1]))
    print(f"{len(prompts)} prompts, {len(todo)} new; judging starts now (a progress line prints every 10 batches)", flush=True)
    import time; t0 = time.time()
    for b, i in enumerate(range(0, len(todo), JBATCH)):
        if (time.time() - START_TIME) / 3600 > MAX_RUN_HOURS:
            HALT = True
            if not DRY_RUN: save_cache(cache, CACHE_PATH)
            raise TimeBudget(f"{MAX_RUN_HOURS} h reached after {i} of {len(todo)} new prompts of this step; cache saved ({len(cache)} judgements)")
        ch = todo[i:i+JBATCH]
        for p, v in zip(ch, _score_chunk(ch)): cache[_key(*p)] = float(v)
        if b % 10 == 0 and not DRY_RUN:
            save_cache(cache, CACHE_PATH)
            if AUTO_DOWNLOAD_EVERY_MIN and (time.time() - globals().get('_last_dl', START_TIME)) / 60 >= AUTO_DOWNLOAD_EVERY_MIN:
                try:
                    from google.colab import files; files.download(CACHE_PATH); globals()['_last_dl'] = time.time()
                except Exception as e: print("auto-download failed:", e)
            done = min(i + JBATCH, len(todo)); el = (time.time() - t0) / 60
            print(f"  {done}/{len(todo)} new prompts done | {done/max(el,1e-9):.0f} per minute | cache size {len(cache)} | elapsed {el:.1f} min", flush=True)
    if not DRY_RUN: save_cache(cache, CACHE_PATH)
    return [cache[_key(*p)] for p in prompts]

SYS = "You are a careful medical evidence reviewer. Answer with a single word: Yes or No."
QUESTIONS = {
 "rel":   "Is the passage relevant to the claim, meaning it addresses the same medical question?",
 "con":   "Does the passage contradict the claim, meaning it says something that cannot be true if the claim is true?",
 "sup":   "Does the passage support the claim?",
 "trust": "Would a careful clinician treat this passage as reliable evidence for the claim, that is, relevant to it and consistent with it?"}
def trunc(t, n=MAX_WORDS):
    w = t.split(); return t if len(w) <= n else " ".join(w[:n]) + " ..."
def judge_frame(df, claim_col, passage_col):
    df = df.copy()
    for k, q in QUESTIONS.items():
        P = [(SYS, f"Claim: {c}\n\nPassage: {trunc(p)}\n\n{q} Answer Yes or No.") for c, p in zip(df[claim_col], df[passage_col])]
        df["J_" + k] = run_judge(P)
    df["J_rel_minus_con"] = df.J_rel - df.J_con; df["J_sup_minus_con"] = df.J_sup - df.J_con
    return df
JSC = {"LLM judge: relevance (CRAG/Self-RAG-style critic)": ("J_rel", 1), "LLM judge: reliability critic": ("J_trust", 1),
       "LLM judge: -contradiction": ("J_con", -1), "LLM judge: relevance minus contradiction": ("J_rel_minus_con", 1), "LLM judge: support minus contradiction": ("J_sup_minus_con", 1)}

## 2. PubMedQA (1,000 queries x 4 conditions, claim-given)

In [ ]:
try:
    if HALT: raise TimeBudget('time budget already reached earlier in this session; skipping')
    if RUN_PUBMED and os.path.exists(f"{OUT}/s6a_pubmedqa_auroc.csv"):
        print("RUN_PUBMED: finished in an earlier session, skipping (results are in", OUT, ")")
    elif RUN_PUBMED:
        ensure_file("rav_med_full_results_N1000.csv")
        df = pd.read_csv("rav_med_full_results_N1000.csv")
        pids = list(dict.fromkeys(df.pubid))[:N_QUERIES]; d = df[df.pubid.isin(pids)].copy()
        d = judge_frame(d, "claim", "passage")
        SC = {"Similarity only (S)": ("similarity", 1), "Contradiction only (-C)": ("contradiction", -1), "ERS (S - C)": ("ERS", 1), **JSC}
        tasks = {"CLEAN vs CONTRADICTION": ["CONTRADICTION"], "CLEAN vs DRIFT+NOISE": ["DRIFT", "NOISE"], "CLEAN vs all corrupted (pooled)": ["CONTRADICTION", "DRIFT", "NOISE"]}
        rows = []
        for tn, negs in tasks.items():
            x = d[d.condition.isin(["CLEAN"] + negs)]
            for sn, (col, sg) in SC.items():
                a, lo, hi = auroc_ci((x.condition == "CLEAN").astype(int), sg * x[col], x.pubid)
                rows.append(dict(task=tn, scorer=sn, AUROC=a, lo=lo, hi=hi))
        T_PQ = pd.DataFrame(rows); print(T_PQ.round(3).to_string(index=False))
        T_PQ.to_csv(f"{OUT}/s6a_pubmedqa_auroc.csv", index=False); d.to_csv(f"{OUT}/s6a_pubmedqa_judge_scores.csv", index=False)
except TimeBudget as e:
    print('STOPPED CLEANLY:', e, '| progress is saved; run all cells again (in a new session if needed) to continue.')

## 3. HealthContradict (920 pairs, 81 questions)

In [ ]:
import hashlib
HC_JSONL = os.environ.get("HC_JSONL", "HealthContradict/dataset/dataset_ready.jsonl")
if not os.path.exists(HC_JSONL):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tinaboya/HealthContradict"], check=True)
raw = pd.DataFrame([json.loads(l) for l in open(HC_JSONL, encoding="utf-8")]); raw["topic_id"] = raw["topic_id"].astype(int)
raw["correct_text"]   = np.where(raw.query_stance == "yes", raw.doc_a, raw.doc_b)     # doc_a always has stance yes, doc_b no
raw["incorrect_text"] = np.where(raw.query_stance == "yes", raw.doc_b, raw.doc_a)
did = lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()[:12]
raw["correct_id"] = raw.correct_text.map(did); raw["incorrect_id"] = raw.incorrect_text.map(did)
docs = {}
for col in ["correct", "incorrect"]:
    for i, t in zip(raw[f"{col}_id"], raw[f"{col}_text"]): docs[i] = t
pairs = raw[["instance_id", "topic_id", "query", "query_stance", "correct_id", "incorrect_id"]].copy()
topics = pairs.drop_duplicates("topic_id").set_index("topic_id")[["query", "query_stance"]]
print(f"{len(pairs)} pairs | {len(topics)} questions | {len(docs)} unique documents")

# claims written by the earlier step (upload hc_claims.csv from the repository: data/healthcontradict/hc_claims.csv)
ensure_file("hc_claims.csv")
claims = pd.read_csv("hc_claims.csv").set_index("topic_id")
assert set(claims.index) == set(topics.index), "hc_claims.csv does not match the 81 questions"
CHUNK_WORDS, MIN_CHUNK_WORDS, MAX_CHUNKS = 80, 15, 120
def chunk(text):
    w = text.split(); ch = [" ".join(w[i:i+CHUNK_WORDS]) for i in range(0, len(w), CHUNK_WORDS)]
    if len(ch) > 1 and len(ch[-1].split()) < MIN_CHUNK_WORDS: ch = ch[:-1]
    return ch[:MAX_CHUNKS] if ch else [text]
need = pd.concat([pairs[["topic_id"]].assign(doc_id=pairs.correct_id), pairs[["topic_id"]].assign(doc_id=pairs.incorrect_id)]).drop_duplicates().reset_index(drop=True)
doc_ids = sorted(need.doc_id.unique())
chunks = pd.DataFrame([(d, k, c) for d in doc_ids for k, c in enumerate(chunk(docs[d]))], columns=["doc_id", "k", "text"])
chunk_emb = embed(chunks.text.tolist())
chunk_index = chunks.reset_index().groupby("doc_id")["index"].apply(list).to_dict()
print(f"{len(doc_ids)} documents -> {len(chunks)} windows")

def top_windows(claim_type):
    """The most claim-similar 80-word window of every (question, document): what a retriever would hand to a generator."""
    rows = []
    for t, d in need.itertuples(index=False):
        ce = embed([claims.loc[t, f"claim_{claim_type}"]])[0]
        ids = chunk_index[d]; S = chunk_emb[ids] @ ce; j = int(np.argmax(S))
        rows.append(dict(topic_id=t, doc_id=d, claim_type=claim_type, window_index=j, S=float(S[j]), window=chunks.text[ids[j]], claim=claims.loc[t, f"claim_{claim_type}"]))
    return pd.DataFrame(rows)

In [ ]:
try:
    if HALT: raise TimeBudget('time budget already reached earlier in this session; skipping')
    if RUN_HC and os.path.exists(f"{OUT}/s6a_healthcontradict_pairwise.csv"):
        print("RUN_HC: finished in an earlier session, skipping (results are in", OUT, ")")
    elif RUN_HC:
        tw = pd.concat([top_windows("ref"), top_windows("free")], ignore_index=True)
        tw["C"] = contra(tw.window.tolist(), tw.claim.tolist()); tw["ERS"] = tw.S - tw.C
        if JUDGE_ALL_WINDOWS:
            print("JUDGE_ALL_WINDOWS is not implemented in this notebook version; using the ERS-selected window.")
        tw = judge_frame(tw, "claim", "window")
        SCH = {"Similarity only (S)": ("S", 1), "Contradiction only (-C)": ("C", -1), "ERS (S - C)": ("ERS", 1), **JSC}
        key = tw.set_index(["topic_id", "doc_id", "claim_type"])
        rows = []
        for ct, ctn in [("ref", "reference claim"), ("free", "label-free claim")]:
            for sn, (col, sg) in SCH.items():
                a = sg * key.loc[list(zip(pairs.topic_id, pairs.correct_id, [ct] * len(pairs))), col].values
                b = sg * key.loc[list(zip(pairs.topic_id, pairs.incorrect_id, [ct] * len(pairs))), col].values
                m, lo, hi = pairwise_ci(win(a, b), pairs.topic_id.values)
                rows.append(dict(claim=ctn, scorer=sn, pairwise_acc=m, lo=lo, hi=hi))
        T_HC = pd.DataFrame(rows); print(T_HC.round(3).to_string(index=False))
        T_HC.to_csv(f"{OUT}/s6a_healthcontradict_pairwise.csv", index=False); tw.drop(columns=["window"]).to_csv(f"{OUT}/s6a_healthcontradict_judge_scores_nowindowtext.csv", index=False)
except TimeBudget as e:
    print('STOPPED CLEANLY:', e, '| progress is saved; run all cells again (in a new session if needed) to continue.')

## 4. SciFact (naturally written claims; see Step 6d for the benchmark description)

In [ ]:
try:
    if HALT: raise TimeBudget('time budget already reached earlier in this session; skipping')
    import tarfile, urllib.request
    SF_DIR = "scifact_data"
    def _download_scifact():
        urllib.request.urlretrieve("https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz", "scifact.tar.gz")
        with tarfile.open("scifact.tar.gz") as t: t.extractall(".")
        if os.path.isdir("data") and not os.path.isdir(SF_DIR): os.rename("data", SF_DIR)
    def _fake_scifact():
        """Tiny made-up SciFact-style files, used only if DRY_RUN is True and the real download is impossible."""
        os.makedirs(SF_DIR, exist_ok=True); r0 = np.random.default_rng(0)
        cp = [dict(doc_id=i, title=f"Title {i}", abstract=[f"Sentence {k} on topic {i%7} " + " ".join(map(str, r0.integers(0, 99, 30))) for k in range(6)], structured=False) for i in range(300)]
        cl = [dict(id=c, claim=f"Claim {c} about topic {c%7}", evidence=({str(c): [dict(sentences=[0], label="SUPPORT")]} if c % 3 == 0 else {str(c): [dict(sentences=[1], label="CONTRADICT")]} if c % 3 == 1 else {}), cited_doc_ids=[c, (c + 1) % 300]) for c in range(120)]
        open(f"{SF_DIR}/corpus.jsonl", "w").write("\n".join(map(json.dumps, cp)))
        open(f"{SF_DIR}/claims_train.jsonl", "w").write("\n".join(map(json.dumps, cl[:90]))); open(f"{SF_DIR}/claims_dev.jsonl", "w").write("\n".join(map(json.dumps, cl[90:])))
    if not os.path.exists(f"{SF_DIR}/corpus.jsonl"):
        try: _download_scifact()
        except Exception as e:
            if not DRY_RUN: raise
            print("Real SciFact download failed in dry run; using tiny fake data:", e); _fake_scifact()
    rd = lambda f: [json.loads(l) for l in open(f"{SF_DIR}/{f}", encoding="utf-8")]
    corpus = {int(d["doc_id"]): (d["title"] + ". " + " ".join(d["abstract"])).strip() for d in rd("corpus.jsonl")}
    claims_sf = rd("claims_train.jsonl") + rd("claims_dev.jsonl")          # the test split has no public labels
    rows, mixed = [], 0
    for c in claims_sf:
        ev = c.get("evidence", {})
        for d, lst in ev.items():
            labs = {e["label"] for e in lst}
            if len(labs) == 1: rows.append((c["id"], c["claim"], int(d), "SUPPORT" if "SUPPORT" in labs else "CONTRADICT"))
            else: mixed += 1
        if not ev:
            for d in c.get("cited_doc_ids", []): rows.append((c["id"], c["claim"], int(d), "NEI_ONTOPIC"))
    sf = pd.DataFrame(rows, columns=["claim_id", "claim", "doc_id", "cond"])
    # irrelevant control: a random abstract the claim does not cite
    allids = np.array(sorted(corpus)); r2 = np.random.default_rng(SEED); irr = []
    cited = {c["id"]: set(map(int, c.get("cited_doc_ids", []))) | set(map(int, c.get("evidence", {}).keys())) for c in claims_sf}
    for cid, g in sf[sf.cond.isin(["SUPPORT", "CONTRADICT"])].groupby("claim_id"):
        while True:
            d = int(r2.choice(allids))
            if d not in cited[cid]: break
        irr.append((cid, g.claim.iloc[0], d, "IRRELEVANT"))
    sf = pd.concat([sf, pd.DataFrame(irr, columns=sf.columns)], ignore_index=True).drop_duplicates(["claim_id", "doc_id"]).reset_index(drop=True)
    sf = sf[sf.doc_id.isin(corpus)].copy(); sf["passage"] = sf.doc_id.map(corpus)
    print("SciFact items by condition:", sf.cond.value_counts().to_dict(), "| claims:", sf.claim_id.nunique(), "| docs with mixed labels dropped:", mixed)

    def _wins(text):
        w = text.split(); ch = [" ".join(w[i:i+80]) for i in range(0, len(w), 80)]
        if len(ch) > 1 and len(ch[-1].split()) < 15: ch = ch[:-1]
        return ch[:120] if ch else [text]

    def score_scifact(df, mode):
        """mode 'full': whole abstract as one passage (as in PubMedQA); 'win': most claim-similar 80-word window (as in HealthContradict)."""
        if mode == "full":
            P = df.passage.tolist(); H = df.claim.tolist()
            S = cos(P, H); C = contra(P, H)
        else:
            wins, owner = [], []
            for i, p in enumerate(df.passage.tolist()):
                for w in _wins(p): wins.append(w); owner.append(i)
            owner = np.array(owner); WE = embed(wins); uc = df.claim.unique().tolist(); CE = dict(zip(uc, embed(uc)))
            best = {}
            for k, (i, w) in enumerate(zip(owner, wins)):
                s = float(WE[k] @ CE[df.claim.iloc[i]])
                if i not in best or s > best[i][0]: best[i] = (s, w)
            S = np.array([best[i][0] for i in range(len(df))]); C = contra([best[i][1] for i in range(len(df))], df.claim.tolist())
        out = df.copy(); out["S"] = S; out["C"] = C; out["ERS"] = S - C
        return out

    def scifact_tables(scored, scorers, tag):
        tasks = {"SUPPORT vs CONTRADICT": ["CONTRADICT"], "SUPPORT vs IRRELEVANT": ["IRRELEVANT"], "SUPPORT vs NEI_ONTOPIC": ["NEI_ONTOPIC"], "SUPPORT vs all others (pooled)": ["CONTRADICT", "IRRELEVANT", "NEI_ONTOPIC"]}
        rows = []
        for tn, negs in tasks.items():
            x = scored[scored.cond.isin(["SUPPORT"] + negs)]
            if x.cond.nunique() < 2: continue
            for sn, (col, sg) in scorers.items():
                a, lo, hi = auroc_ci((x.cond == "SUPPORT").astype(int), sg * x[col], x.claim_id)
                rows.append(dict(passage_unit=tag, task=tn, scorer=sn, n_pos=int((x.cond == "SUPPORT").sum()), n_neg=int((x.cond != "SUPPORT").sum()), AUROC=a, lo=lo, hi=hi))
        return pd.DataFrame(rows)

    if RUN_SCIFACT and os.path.exists(f"{OUT}/s6a_scifact_auroc.csv"):
        print("RUN_SCIFACT: finished in an earlier session, skipping (results are in", OUT, ")")
    elif RUN_SCIFACT:
        sc_ = score_scifact(sf, "full")
        sc_ = judge_frame(sc_, "claim", "passage")
        SCF = {"Similarity only (S)": ("S", 1), "Contradiction only (-C)": ("C", -1), "ERS (S - C)": ("ERS", 1), **JSC}
        T_SF = scifact_tables(sc_, SCF, "full abstract"); print(T_SF.round(3).to_string(index=False))
        T_SF.to_csv(f"{OUT}/s6a_scifact_auroc.csv", index=False)
except TimeBudget as e:
    print('STOPPED CLEANLY:', e, '| progress is saved; run all cells again (in a new session if needed) to continue.')

## 5. Save

In [ ]:
if os.path.exists(CACHE_PATH): shutil.copy(CACHE_PATH, OUT)   # the progress file travels with the results

import shutil
shutil.make_archive("rav_med_step6a_outputs", "zip", OUT)
try:
    from google.colab import files; files.download("rav_med_step6a_outputs.zip")
except ImportError: pass
print("Saved rav_med_step6a_outputs.zip")